# 01 - Qualite des donnees et analyse descriptive

**Objectif** : documenter l'etat des donnees brutes (Bronze), justifier chaque regle de nettoyage appliquee en Silver, presenter les resultats Great Expectations, puis decrire les donnees propres.

Prerequis : stack Docker demarree, DAGs `10`, `12`, `13`, `20` termines. Executer *Run All*.

In [ ]:
%pip install -q pandas pyarrow boto3 matplotlib requests "elasticsearch==8.13.2" scipy
import io, json, re, sys, collections
import boto3, pandas as pd, matplotlib.pyplot as plt, requests
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 160)
sys.path.insert(0, "../src")          # modules de normalisation du projet (purs, sans dependance Docker)

# Acces MinIO depuis votre machine (la stack Docker doit tourner)
s3 = boto3.client("s3", endpoint_url="http://localhost:9000", aws_access_key_id="minioadmin",
                  aws_secret_access_key="minioadmin123", region_name="us-east-1")

def list_keys(bucket, prefix=""):
    out, token = [], None
    while True:
        kw = dict(Bucket=bucket, Prefix=prefix, **({"ContinuationToken": token} if token else {}))
        r = s3.list_objects_v2(**kw); out += [o["Key"] for o in r.get("Contents", [])]
        if not r.get("IsTruncated"): return out
        token = r["NextContinuationToken"]

def read_jsonl(bucket, keys):
    rows = []
    for k in keys:
        body = s3.get_object(Bucket=bucket, Key=k)["Body"].read().decode("utf-8")
        rows += [json.loads(l) for l in body.splitlines() if l]
    return pd.DataFrame(rows)

def latest_partition(bucket):
    keys = sorted(k for k in list_keys(bucket) if k.endswith(".jsonl") and k.startswith("dt="))
    return [k for k in keys if k.startswith(keys[-1].split("/")[0] + "/")]

def read_silver(bucket):
    return pd.read_parquet(io.BytesIO(s3.get_object(Bucket=bucket, Key="data/part-0000.parquet")["Body"].read()))

def read_report(bucket, key="_reports/latest.json"):
    return json.loads(s3.get_object(Bucket=bucket, Key=key)["Body"].read())


## 1. Chargement du Bronze (donnees brutes)

In [ ]:
arx_raw = read_jsonl("arxiv-raw", latest_partition("arxiv-raw"))
hn_raw  = read_jsonl("hn-raw", [k for k in list_keys("hn-raw") if k.endswith(".jsonl")])
cit_raw = read_jsonl("citations-raw", latest_partition("citations-raw"))
print(f"ArXiv : {len(arx_raw):,} | Hacker News : {len(hn_raw):,} | OpenAlex : {len(cit_raw):,}")

## 2. Anomalies du Bronze : ArXiv

In [ ]:
def null_rate(df):
    return (df.isna() | (df.astype(str).str.strip() == "")).mean().mul(100).round(2).sort_values(ascending=False)

print("Taux de valeurs manquantes ou vides (%), ArXiv brut :"); display(null_rate(arx_raw[["id","title","abstract","categories","doi","journal-ref","comments","versions","update_date"]]))
nl_title = arx_raw["title"].str.contains(r"\n", regex=True, na=False).mean() * 100
nl_abs   = arx_raw["abstract"].str.contains(r"\n", regex=True, na=False).mean() * 100
multi_sp = arx_raw["title"].str.contains(r"\s{2,}", regex=True, na=False).mean() * 100
print(f"\nTitres avec retour a la ligne : {nl_title:.1f} % | resumes : {nl_abs:.1f} % | titres avec espaces multiples : {multi_sp:.1f} %")
print(f"Doublons sur `id` : {arx_raw['id'].duplicated().sum()}")
print("Exemple de champ `categories` :", arx_raw["categories"].iloc[0], "->", type(arx_raw["categories"].iloc[0]).__name__, "(une seule chaine)")
print("Exemple de date de version :", arx_raw["versions"].iloc[0][0]["created"], "(chaine RFC-2822)")
print("Articles sans aucune version :", arx_raw["versions"].map(lambda v: not v).sum())

**Constats -> regles Silver** : titres/resumes avec retours a la ligne -> normalisation des espaces ; `categories` en une chaine -> liste + categorie principale ; dates en chaine RFC-2822 -> timestamp UTC ; absence de date -> annee deduite de l'identifiant.

## 3. Anomalies du Bronze : Hacker News

In [ ]:
print("Enregistrements par source :"); display(hn_raw["_source"].value_counts())
cols = {s: sorted(c for c in hn_raw[hn_raw["_source"] == s].dropna(axis=1, how="all").columns if not c.startswith("_")) for s in hn_raw["_source"].unique()}
for s, c in cols.items(): print(f"\nChamps de la source {s} : {c}")
print("\nChamps de score : Firebase = `score`, Algolia = `points` -> deux schemas a harmoniser")

from scipulse.normalize import arxiv_id_from_url
urls = hn_raw["url"].dropna()
print("\nFormes d'URL ArXiv rencontrees :")
shapes = urls[urls.str.contains("arxiv", case=False)].str.extract(r"arxiv[^/]*\.org/(\w+)/")[0].value_counts(); display(shapes)
print(f"URL vides : {hn_raw['url'].isna().mean()*100:.1f} % | URL avec identifiant ArXiv extractible : {urls.map(arxiv_id_from_url).notna().sum():,}")
hn_raw["_id"] = hn_raw["story_id"].fillna(hn_raw["id"]) if "story_id" in hn_raw else hn_raw["id"]
print(f"Meme story presente plusieurs fois : {hn_raw['_id'].duplicated().sum():,} lignes dupliquees (le score evolue entre deux lectures)")

**Constats -> regles Silver** : deux schemas -> schema unique ; identifiant ArXiv enfoui dans l'URL (`/abs/`, `/pdf/...pdf`, suffixe de version) -> extraction par expression reguliere ; doublons -> on garde la lecture la plus recente ; commentaires/jobs melanges -> on garde les `story`.

## 4. Anomalies du Bronze : OpenAlex

In [ ]:
from scipulse.normalize import arxiv_id_from_doi
is_arxiv_doi = cit_raw["doi"].map(arxiv_id_from_doi).notna()
print(f"Works dont le champ `doi` est le DOI ArXiv : {is_arxiv_doi.mean()*100:.1f} %  ->  les autres portent le DOI de la revue")
print("Exemples de DOI :"); display(cit_raw["doi"].sample(5, random_state=1).to_frame())
if "locations" in cit_raw:
    def has_arxiv_loc(locs):
        return any(isinstance(l, dict) and "arxiv.org" in str(l.get("landing_page_url", "")) for l in (locs or []))
    print(f"Works avec une localisation arxiv.org : {cit_raw['locations'].map(has_arxiv_loc).mean()*100:.1f} %  (c'est par la que l'identifiant ArXiv est retrouve)")
print("Casse / prefixe des DOI :", cit_raw["doi"].dropna().str.startswith("https://doi.org/").mean()*100, "% avec prefixe https://doi.org/")
print("Valeurs manquantes (%) :"); display(null_rate(cit_raw[["doi","title","publication_year","cited_by_count","primary_topic"]]))

**Constats -> regles Silver** : le `doi` OpenAlex n'est pas l'identifiant ArXiv -> jointure via `locations` ; DOI avec prefixe et casse variable -> normalisation (minuscule, sans prefixe, comme demande par le brief) ; works sans version ArXiv -> rejetes.

## 5. Resultats du nettoyage et validation Great Expectations

In [ ]:
rows = []
for src, b in (("arxiv", "arxiv-clean"), ("hn", "hn-clean"), ("citations", "citations-clean")):
    r = read_report(b); ge = r["great_expectations"]
    rows.append({"source": src, "bronze": r["raw"], "silver": r["kept"], "rejetes": r["raw"] - r["kept"],
                 "doublons_supprimes": r.get("duplicates_removed", 0), "GE_expectations": ge["evaluated"], "GE_reussies": ge["passed"],
                 "GE_succes_%": ge["success_percent"]})
summary = pd.DataFrame(rows); display(summary)
for src, b in (("arxiv", "arxiv-clean"), ("hn", "hn-clean"), ("citations", "citations-clean")):
    failed = read_report(b)["great_expectations"]["failed"]
    print(f"\n[{src}] expectations en echec :", failed if failed else "aucune")
for src, b in (("arxiv", "arxiv-clean"), ("hn", "hn-clean"), ("citations", "citations-clean")):
    r = read_report(b); print(f"\n[{src}] details du nettoyage :", {k: v for k, v in r.items() if k not in ("great_expectations","columns","source","path","run_at")})

## 6. Statistiques descriptives (Silver)

In [ ]:
arx, hn, cit = read_silver("arxiv-clean"), read_silver("hn-clean"), read_silver("citations-clean")
fig, ax = plt.subplots(2, 2, figsize=(14, 9))
arx.groupby("year").size().plot(ax=ax[0,0], marker="o", title="Articles ArXiv (echantillon) par annee"); ax[0,0].set_ylabel("articles")
arx["primary_category"].value_counts().head(15).sort_values().plot.barh(ax=ax[0,1], title="Top 15 categories principales")
arx["n_authors"].clip(upper=20).plot.hist(bins=20, ax=ax[1,0], title="Nombre d'auteurs par article (plafonne a 20)")
arx["abstract"].str.split().str.len().plot.hist(bins=40, ax=ax[1,1], title="Longueur des resumes (mots)")
plt.tight_layout(); plt.show()
print(arx[["n_authors","n_versions","year"]].describe().round(2))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
hn["score"].clip(lower=1).plot.hist(bins=40, ax=ax[0], log=True, title="HN : score des stories (echelle log)")
hn[hn.arxiv_id.notna()].groupby("year").size().plot.bar(ax=ax[1], title="Stories HN pointant vers ArXiv, par annee")
hn["domain"].value_counts().head(10).sort_values().plot.barh(ax=ax[2], title="Top 10 domaines cites sur HN")
plt.tight_layout(); plt.show()
print(f"Stories HN : {len(hn):,} dont {hn.arxiv_id.notna().sum():,} pointant vers un article ArXiv ({hn.arxiv_id.nunique():,} articles distincts)")
print(hn[["score","n_comments"]].describe().round(1))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
cit["cited_by_count"].clip(lower=1).plot.hist(bins=50, ax=ax[0], log=True, title="Citations par article (echelle log)")
cit["field"].value_counts().head(8).sort_values().plot.barh(ax=ax[1], title="Domaine disciplinaire principal (OpenAlex)")
plt.tight_layout(); plt.show()
print(cit[["cited_by_count","n_references"]].describe().round(1))
share = arx.arxiv_id.isin(cit.arxiv_id).mean() * 100
print(f"\nArticles ArXiv avec donnees de citation : {share:.1f} %")
print(f"Articles ArXiv mentionnes sur HN : {arx.arxiv_id.isin(hn.arxiv_id.dropna()).mean()*100:.2f} %")

## 7. Conclusions

A completer apres execution : (1) quelles anomalies pesent le plus sur le volume ; (2) taux de couverture de la jointure ; (3) distributions fortement asymetriques (citations, scores HN) qui justifient l'echelle logarithmique du score d'impact.